<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Choosing what to apply: candidates from a frame stream →](02_Choosing_What_To_Apply.ipynb)

# 1 · Labelling a shelf: a transform, applied and certified

**The problem.** A generator that rewrites a model makes choices: what to name a thing, which form to give it, what to
leave out. Made silently in its code, they can't be reviewed, compared between two runs, or made again the same way.
Here the model is a shelf of parts, some without a label, and the rewrite labels each with its name, in upper or lower
case. The case is the choice: someone must decide it, item by item, and each decision must be kept.

mbse-transforms makes such a rewrite a sequence of steps, each choice in it a parameter that someone decides. As the
user put it, "A transform has a prototype, a procedure, and a cerificate." This case study builds one of each:

1. the **prototype**, a pattern, here an item without a label, which declares the transform's parameters (1.1);
2. the **procedure**, native code registered by name, called with the caller's visitors and the parameter assignments
   (1.2);
3. the **application**, `Transforms.apply`, then its **certification**, `Transforms.certify`, two calls (1.3);
4. the **certificates** they write, read back as any data is (1.4).

The shelf's schemas are in `toolkit`, with the helpers the case studies share: `shelf.Shelf`, a singleton owning
`shelf.Item`s by key, each with a `name` and a `label`; and `shelf.Case`, a union of two options, `upper` and `lower`,
each an empty value object named by its branch.

In [1]:
import { Builders, Parametrics, Plain, Proxies } from "@mbse/schemas/Framework";
import { Transforms } from "@mbse/transforms";

import { assignments_of, bear, frames, items, shelf, shown } from "./toolkit.js";

/** shelf.Unlabelled, an item without a label, declaring the parameter case, a shelf.Case. */
function patterns(txn: any): any {
  return txn.uid("tutorial.patterns/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Unlabelled: txn.Schemas.OfPattern })
    .spec([
      (_: any) => _.Unlabelled.name("shelf.Unlabelled").symbols({ i: txn.shelf.Item })
        .spec([(f: any) => f.i.label(null)])
        .parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          case: (d: any) => d.parameter((p: any) => p.description("the label's case").for_value([txn.shelf.Case])) }))]),
    ])]))]);
}

/** A store of the shelf's schemas and its prototype, its shelf holding an item of each name, keyed by it. */
function model(...names: string[]): any {
  const store: any = new Builders.Store();
  store.transaction(Parametrics.install);
  store.transaction(shelf);
  store.transaction(patterns);
  store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ s: t.shelf.Shelf })]))]));
  for (const name of names) {
    store.transaction(() => {
      store.shelf.Shelf(store.singleton("shelf.TheShelf"), (s: any) => s.items({
        [name]: (e: any) => e.item((i: any) => i.name(name)) }));
    });
  }
  return store;
}

`shelf.Unlabelled` is the prototype: a pattern of one symbol, `i`, an item whose `label` is absent. A pattern
instance is a frame: each frame of `shelf.Unlabelled` binds `i` to one unlabelled item. The pattern declares the
transform's parameter, `case`, through mbse-schemas' `Parametrics`: a value of `shelf.Case`, which someone will decide
for each frame. `model` makes a store of the shelf's schemas and the prototype, holding an item of each name given.

<a id="T01.1"></a>
## 01.1 · A transform is data: its prototype and the name of its procedure

A transform is mbse-schemas data, written, read and compared as any data is, with no code of its own. In the store
you keep your transforms in, with your schemas and patterns, `Transforms.install` installs its schemas, and
`Transforms.define(txn, name, procedure, prototype)` writes a transform: owned by the catalog,
`Transforms.TheCatalog`, under its name, holding its prototype and the name its procedure is registered under. Its
parameters are those its prototype declares. Run again, either finds what it wrote and writes nothing.

In [2]:
const transforms: any = model(); // the transforms' store: the shelf's schemas and the prototype, and no items
transforms.transaction(Transforms.install);
transforms.transaction((t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled));
transforms.transaction(() => {
  for (const entry of transforms.singleton("Transforms.TheCatalog").transforms) {
    const [prototype] = [...entry.transform.prototype].map((e: any) => e.pattern);
    const [declared] = [...new Builders.OfPattern.Builder(prototype, false, transforms).parameters];
    const parameters = [...declared.params.parameters].map((e: any) => String(e.name)).join(", ");
    console.log(`${entry.name}: prototype ${prototype.name}, procedure ${entry.transform.procedure}, parameters ${parameters}`);
  }
});

Label: prototype shelf.Unlabelled, procedure label, parameters case


<a id="T01.2"></a>
## 01.2 · A procedure decides nothing

A procedure is a native function, `procedure(source, sync, target, assignments)`, registered by name in a
`Transforms.Procedures` you make (1.3). It is called with the visitors you give and the
parameter assignments, read-only, and that's that: as the user put it, "procedures (such as validation,
simplification, comparison, etc.) take visitor(s) as an argument(s) and that's that. No additional criteria or
invention beyond the visitor boundary is acceptable." An argument you don't give is `null`.

It decides nothing: every choice is a parameter, taken from `assignments`. "Transforms may not make decisions below
the parameter level. Discrete decisions in a transformation procedure must themselves be modeled as discrete
transforms with non-overlapping patterns." `label` reads the sync's item, the object its frame binds to `i`, and
writes its label through the item's store's builder, in the case its assignment gives.

In [3]:
/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const item = sync.i;
  const name = String(item.name);
  (Proxies.store_of(item) as any).shelf.Item(item, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase()));
}

<a id="T01.3"></a>
## 01.3 · Apply, then certify, with your own visitors

An application takes up to three arguments, each a `Visitors.OfPattern` you make, bearing its own pattern frame, in
the user's words: "`source`- a read-only `Visitors.OfPattern` bearing a pattern frame bound to a source copy that is not
to be disturbed (nondestructive)", "`sync`- a writeable `Visitors.OfPattern` bearing a pattern frame bound to a copy
that is to be edited in place (destructive)", and "`target`- a writeable `Visitors.OfPattern` bearing a pattern frame
bound to a (usually empty) copy that is to be replaced (constructive)". "The transforms application procedure does not
need to know how copies are made since it takes Visitors.OfPattern as an argument." Here the shelf is edited in place:
each item's frame is the sync.

You find the frames by a query of the data's store (`toolkit.frames`), and, within your transactions, make each one's
visitor (`toolkit.bear`, which finds each object a frame binds again: the query gives its frames back after its
transaction ends). Then, for each frame:

- `Transforms.apply(transform, assignments, procedures, { sync })` calls the procedure, and that's that: it writes
  nothing and opens no transaction; `transform` is the visitor of the transform, read within your transaction;
- `Transforms.certify(store, transform, assignments, history, by, { sync })`, a follow-up call with the same
  arguments, writes the certificates in `store`, the certificates' store, in the one transaction mbse-transforms opens,
  appended to the history named `history`, recording that `by` decided the parameter assignments. You hold no
  transaction open on `store` while it does.

An application's arguments are an options object in TypeScript. You decide each item's case yourself here; case study
2 takes the choices from a frame stream. Your transactions commit what the procedure wrote, or roll it back.

In [4]:
const procedures = new Transforms.Procedures(); // a registry is a value you make: none is global
procedures.register("label", label);
const data: any = model("bolt", "nut"); // the copy edited in place
const certs: any = new Builders.Store(); // the certificates' store
certs.transaction(Transforms.install);
const choices: Record<string, string> = { bolt: "upper", nut: "lower" }; // your decisions, one per item
const found = frames(data, "shelf.Unlabelled");
transforms.transaction(() => data.transaction(() => { // your transactions
  const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
  for (const frame of found) {
    const sync = bear(data, "shelf.Unlabelled", frame);
    const assignments = { case: choices[String(sync.i.name)] };
    Transforms.apply(Label, assignments, procedures, { sync });
    Transforms.certify(certs, Label, assignments, "run", "caller", { sync });
  }
}));
items(data);

bolt BOLT
nut nut


<a id="T01.4"></a>
## 01.4 · The certificates, read back

"There's actually 3 levels of certificates that form a complete audit history for transforms":

- the **history certificate**, "which records a sequence of transforms", owned by `Transforms.TheHistories` under its
  name: its transform certificates in order (`steps`, by `index`), and who decided each (`decisions`, by `by`);
- a **transform certificate**, "which records the application, linked to the transform's prototype, the history
  certificate, and prev/next transforms in the history": its parameter assignments ("ofc those parameters must be
  recorded in the transform certificate"), its transform and prototype, each by object reference, and `prev` and
  `next`;
- a **sub-certificate** "for the source, sync, and target (whichever are present)", under its argument: the frame its
  visitor bore, by object reference, the pattern it is a frame of, and the transform certificate before it on the same
  pattern instance in the history, if any.

They are read within a transaction of the certificates' store, as any data is:

In [5]:
certs.transaction(() => {
  const history = certs.singleton("Transforms.TheHistories").histories.at("run").history;
  const decisions = [...history.decisions];
  for (const [k, entry] of [...history.steps].entries()) {
    const step = entry.step;
    const args = [...step.frames].map((e: any) => String(e.argument)).join(", ");
    const prev = [...step.prev].flatMap((e: any) => [...e.before.history].map((h: any) => `step ${h.index.get()}`)).join(", ") || "none";
    console.log(`step ${entry.index.get()}: ${shown(assignments_of(step))}, decided by ${decisions[k].by}, arguments: ${args}, prev: ${prev}`);
  }
});

step 0: case=upper, decided by caller, arguments: sync, prev: none
step 1: case=lower, decided by caller, arguments: sync, prev: step 0


The certificates are data, so mbse-schemas' own serialization writes and reads them, with no code of
mbse-transforms': "No model should need bespoke schema-aware code to read or write. Use mbse-schemas built-in
serialization support." A snapshot of the certificates' store, read into a store of its own, holds the same history:

In [6]:
const log = Plain.ToPlain(certs).Snapshot(); // the certificates' store, as plain data
const back: any = new Builders.Store();
Plain.FromPlain(back).Snapshot(log); // read back into a store of its own
back.transaction(() => {
  const history = back.singleton("Transforms.TheHistories").histories.at("run").history;
  console.log([...history.steps].map((e: any) => shown(assignments_of(e.step))).join("; "));
});

case=upper; case=lower


Case study 2 takes the candidates from a frame stream, ranked by a policy, rather than deciding each frame by
hand.

<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Choosing what to apply: candidates from a frame stream →](02_Choosing_What_To_Apply.ipynb)